# Coupling a symbolic half-step to a numerical diagnostic
### Porous-channel flow with `asymptotics` + SciPy

This tutorial weaves the `asymptotics` library into a larger investigation: we take
an order equation the symbolic solver *cannot* finish, solve **that** equation
numerically with SciPy, and compare the resulting two-term expansion against a fully
numerical solution of the original problem.

The problem is the self-similar porous-channel flow
$$\varepsilon F''' + F F'' - (F')^2 = \lambda,\qquad
  F(0)=0,\ F'(1)=0,\ F(1)=1,\ F''(0)=0,$$
with $\varepsilon = 1/R$ the inverse cross-flow Reynolds number and $\lambda$ a
space-invariant eigenvalue determined together with $F$; the fourth
(centreline-symmetry) condition $F''(0)=0$ closes it. Both quantities are expanded,
$F=F_0+\varepsilon F_1+\cdots$ and $\lambda=\lambda_0+\varepsilon\lambda_1+\cdots$.

In [1]:
import numpy as np, sympy as sp
import matplotlib.pyplot as plt
from scipy.integrate import solve_bvp
from asymptotics import ODE
pi = np.pi          # numerical pi for the SciPy work; use sp.pi for symbolic work

## 1. Symbolic half-steps

The leading-order equation $F_0 F_0'' - (F_0')^2 = \lambda_0$ has the known
Taylor--Yuan solution $F_0=\sin(\pi y/2)$ with $\lambda_0=-\pi^2/4$ (direct
substitution gives $-\pi^2/4$). We supply the known $\lambda_0$ to the library
(as `+ pi**2/4`) and **declare $\lambda_1$ as a symbolic parameter** (`- eps*lambda1`),
so the library carries the $O(\varepsilon)$ eigenvalue correction into the order-1
equation. It does not compute $\lambda_1$ (its value follows from a regularity
condition, below), but `sol[1].ode` is then *exactly* the displayed order-$\varepsilon$
equation. Note `sp.pi` in the symbolic solution so the residual check simplifies to
zero exactly.

In [2]:
y = sp.Symbol('y')
eq = ODE("eps*F''' + F*F'' - F'**2 + pi**2/4 - eps*lambda1",
         small_param="eps", independent="y",
         conditions=["F(0) = 0", "F'(1) = 0", "F(1) = 1"])
sol = eq.begin_expansion(order=1)
sol[0].set_solution(sp.sin(sp.pi*y/2))   # sp.pi (exact); verified against the equation
sol[1].ode                                # exactly L[F1] = lambda1 + (pi**3/8) cos(pi y/2)

  ℹ️  dependent = 'F' (inferred from conditions), independent = 'y' (supplied)
     To override: ODE(..., dependent='F', independent='y')
  ⚠️  symbolic parameters detected: {'lambda1'}
     Provide values at eval/compare time:
     sol.eval(eps=0.1, at=t_vals, params={'lambda1': value})


  ✓  Order 0 set manually: F_0 = sin(pi*y/2)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

O(ε¹) — symbolic:
  Eq(-lambda1 + F_0(y)*Derivative(F_1(y), (y, 2)) + F_1(y)*Derivative(F_0(y), (y, 2)) - 2*Derivative(F_0(y), y)*Derivative(F_1(y), y) + Derivative(F_0(y), (y, 3)), 0)

O(ε¹) — substituted:
  Eq(-lambda1 - pi**2*F_1(y)*sin(pi*y/2)/4 + sin(pi*y/2)*Derivative(F_1(y), (y, 2)) - pi*cos(pi*y/2)*Derivative(F_1(y), y) - pi**3*cos(pi*y/2)/8, 0)

## 2. Full nonlinear reference (SciPy)

We solve the original third-order nonlinear BVP numerically, imposing all four
conditions including the centreline symmetry $F''(0)=0$, which closes the nonlinear
eigenvalue $\lambda$. `solve_bvp` carries the unknown $\lambda$ as a parameter.

In [3]:
def full(eps):
    def rhs(t, Y, p):
        F, Fp, Fpp = Y; lam = p[0]
        return np.vstack([Fp, Fpp, (lam - F*Fpp + Fp**2)/eps])
    def bc(Ya, Yb, p):
        return np.array([Ya[0], Ya[2], Yb[1], Yb[0]-1.0])   # F(0)=0, F''(0)=0, F'(1)=0, F(1)=1
    t = np.linspace(0, 1, 201)
    Y0 = np.vstack([np.sin(pi*t/2), (pi/2)*np.cos(pi*t/2), -(pi/2)**2*np.sin(pi*t/2)])
    s = solve_bvp(rhs, bc, t, Y0, p=[-pi**2/4], max_nodes=40000, tol=1e-9)
    assert s.success, f'full nonlinear solve failed at eps={eps}: {s.message}'
    return s

## 3. Solve `sol[1].ode` numerically

The order-$\varepsilon$ correction is an eigenproblem for $(F_1,\lambda_1)$. The
centreline $y=0$ is a **singular point** of this equation (the coefficient
$\sin(\pi y/2)$ vanishes); requiring a solution regular there gives, from the
equation evaluated at $y=0$, the relation $-\pi F_1'(0)=\lambda_1+\tfrac{\pi^3}{8}$,
which together with $F_1(0)=0,\ F_1(1)=0,\ F_1'(1)=0$ fixes $(F_1,\lambda_1)$.
Numerically we solve on $[a,1]$ with a small $a$ to stay off the singularity and let
`solve_bvp` carry $\lambda_1$ as an unknown parameter:
$$\sin(\tfrac{\pi y}{2})F_1'' - \pi\cos(\tfrac{\pi y}{2})F_1'
  - \tfrac{\pi^2}{4}\sin(\tfrac{\pi y}{2})F_1
  = \lambda_1 + \tfrac{\pi^3}{8}\cos(\tfrac{\pi y}{2}).$$
(We do **not** impose $F_1''(0)=0$ directly; it emerges from regularity.)

In [4]:
def f1_solve(a=1e-4):
    def rhs(t, Y, p):
        F1, F1p = Y; l1 = p[0]
        s, c = np.sin(pi*t/2), np.cos(pi*t/2)
        return np.vstack([F1p, (l1 + (pi**3/8)*c + pi*c*F1p + (pi**2/4)*s*F1)/s])
    def bc(Ya, Yb, p):
        return np.array([Ya[0], Yb[0], Yb[1]])   # F1(a)=0, F1(1)=0, F1'(1)=0
    t = np.linspace(a, 1, 401)
    s = solve_bvp(rhs, bc, t, np.zeros((2, t.size)), p=[-2.0], max_nodes=60000, tol=1e-7)
    assert s.success, f'O(eps) eigen-BVP failed: {s.message}'
    return s

f1 = f1_solve()
print('eigenvalue correction lambda1 =', round(float(f1.p[0]), 4))
# regularity check: -pi*F1'(0) should equal lambda1 + pi**3/8
print('regularity: -pi*F1\'(0) =', round(-pi*f1.y[1,0], 4),
      ' vs lambda1 + pi**3/8 =', round(float(f1.p[0]) + pi**3/8, 4))

eigenvalue correction lambda1 = -2.0525
regularity: -pi*F1'(0) = 1.8233  vs lambda1 + pi**3/8 = 1.8233


## 4. Compare the two-term expansion to the full solution

The composite $F_0 + \varepsilon F_1$ (with $F_1$ from the numerical half-step) is
compared to the full numerical solution across several $\varepsilon$.

In [5]:
print(f"{'eps':>6} {'||F0-F||':>12} {'||F0+epsF1-F||':>16}")
rows=[]
for eps in [0.02, 0.05, 0.1, 0.2]:
    s = full(eps); t = np.linspace(1e-3, 1, 400); Ff = s.sol(t)[0]
    F0 = np.sin(pi*t/2); comp = F0 + eps*f1.sol(t)[0]
    e0 = np.max(np.abs(F0-Ff)); e1 = np.max(np.abs(comp-Ff)); rows.append((eps,e0,e1))
    print(f"{eps:>6} {e0:>12.3e} {e1:>16.3e}")

   eps     ||F0-F||   ||F0+epsF1-F||


  0.02    2.346e-03        2.145e-04
  0.05    5.195e-03        1.241e-03
   0.1    8.550e-03        4.332e-03
   0.2    1.229e-02        1.339e-02


In [6]:
eps = 0.1; s = full(eps)
t = np.linspace(0,1,300); tt = np.linspace(1e-4,1,300)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].plot(t, s.sol(t)[0], 'k-', lw=2, label='Numerical reference')
ax[0].plot(t, np.sin(pi*t/2), 'C0--', lw=1.5, label='F0')
ax[0].plot(tt, np.sin(pi*tt/2)+eps*f1.sol(tt)[0], 'C3-.', lw=1.5, label='F0 + eps*F1')
ax[0].set_xlabel('y'); ax[0].set_ylabel('F'); ax[0].legend(); ax[0].set_title(f'eps={eps}')
ev=np.array([r[0] for r in rows])
ax[1].loglog(ev,[r[1] for r in rows],'C0o--',label=r'$\|F_0-F\|_\infty$')
ax[1].loglog(ev,[r[2] for r in rows],'C3s-.',label=r'$\|F_0+\varepsilon F_1-F\|_\infty$')
ax[1].set_xlabel('eps'); ax[1].set_ylabel('max error'); ax[1].legend()
plt.tight_layout(); plt.show()

## Takeaways

- `sol[1].ode` is a raw SymPy object (exactly Eq. 8, with the declared symbolic
  parameter $\lambda_1$): it can be fed to `lambdify`, `dsolve`, or a numerical
  solver, or manipulated term by term.
- Adding the numerically computed $F_1$ improves accuracy, and the improvement grows
  as $\varepsilon\to0$ (an $O(\varepsilon^2)$ composite error).
- At $\varepsilon=0.2$ the correction no longer helps -- the diagnostic reveals the
  edge of the asymptotic regime. This is the kind of investigation the library is
  meant to support: symbolic hierarchy management from `asymptotics`, numerics from
  SciPy.